<a href="https://colab.research.google.com/github/rahulsharma-crtl/AI_image_detector/blob/main/AI_image_detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install datasets accelerate scikit-learn matplotlib tqdm

Imports And Config


In [ ]:
import os
import random
import copy
from dataclasses import dataclass

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T

from PIL import Image
from datasets import load_dataset
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm


@dataclass
class Config:
    dataset_id: str = "Rajarshi-Roy-research/Defactify_Image_Dataset"
    image_size: int = 224

    # T4-safe. Increase later if it works.
    train_per_class: int = 800
    val_per_class: int = 200

    batch_size: int = 32
    epochs: int = 8
    learning_rate: float = 2e-4
    seed: int = 42
    save_path: str = "best_real_world_ai_detector.pt"


cfg = Config()

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(cfg.seed)

Download Balanced Real/Fake Samples

In [ ]:
def collect_balanced_samples(split, per_class):
    """
    Label_A:
    0 = real
    1 = synthetic / AI-generated
    """

    ds = load_dataset(
        cfg.dataset_id,
        split=split,
        streaming=True
    )

    ds = ds.shuffle(buffer_size=3000, seed=cfg.seed)

    images = []
    labels = []

    counts = {0: 0, 1: 0}

    pbar = tqdm(total=per_class * 2, desc=f"Collecting {split}")

    for example in ds:
        try:
            label = int(example["Label_A"])

            if label not in [0, 1]:
                continue

            if counts[label] >= per_class:
                continue

            image = example["Image"].convert("RGB")

            images.append(image)
            labels.append(label)

            counts[label] += 1
            pbar.update(1)

            if counts[0] >= per_class and counts[1] >= per_class:
                break

        except Exception:
            continue

    pbar.close()

    print(f"{split} counts:", counts)

    return images, labels


train_images, train_labels = collect_balanced_samples("train", cfg.train_per_class)
val_images, val_labels = collect_balanced_samples("validation", cfg.val_per_class)

print("Train images:", len(train_images))
print("Validation images:", len(val_images))

Dataset And Transforms

In [ ]:
train_transform = T.Compose([
    T.Resize((256, 256)),
    T.RandomResizedCrop(cfg.image_size, scale=(0.75, 1.0)),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(
        brightness=0.15,
        contrast=0.15,
        saturation=0.15,
        hue=0.03
    ),
    T.RandomApply([T.GaussianBlur(kernel_size=3)], p=0.15),
    T.ToTensor(),
    T.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

val_transform = T.Compose([
    T.Resize((cfg.image_size, cfg.image_size)),
    T.ToTensor(),
    T.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])


class RealFakeDataset(Dataset):
    def __init__(self, images, labels, transform):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        image = self.images[index]
        label = self.labels[index]

        image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)


train_dataset = RealFakeDataset(train_images, train_labels, train_transform)
val_dataset = RealFakeDataset(val_images, val_labels, val_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Datasets ready.")

Show Sample Images

In [ ]:
def show_samples(images, labels, count=8):
    plt.figure(figsize=(14, 4))

    for i in range(count):
        plt.subplot(1, count, i + 1)
        plt.imshow(images[i])
        title = "Real" if labels[i] == 0 else "AI"
        plt.title(title)
        plt.axis("off")

    plt.show()


show_samples(train_images, train_labels, count=8)

Build ResNet18 Model

In [ ]:
weights = torchvision.models.ResNet18_Weights.DEFAULT
model = torchvision.models.resnet18(weights=weights)

num_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(0.35),
    nn.Linear(num_features, 2)
)

model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg.learning_rate,
    weight_decay=1e-4
)

scaler = torch.cuda.amp.GradScaler(enabled=(device == "cuda"))

print("Model ready.")

Train

In [ ]:
best_acc = 0.0
best_model_weights = copy.deepcopy(model.state_dict())

for epoch in range(1, cfg.epochs + 1):
    model.train()

    train_loss = 0.0
    train_preds = []
    train_truth = []

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch}/{cfg.epochs}"):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=(device == "cuda")):
            logits = model(images)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item() * images.size(0)

        preds = logits.argmax(dim=1).detach().cpu().tolist()
        truth = labels.detach().cpu().tolist()

        train_preds.extend(preds)
        train_truth.extend(truth)

    train_loss = train_loss / len(train_dataset)
    train_acc = accuracy_score(train_truth, train_preds)

    model.eval()

    val_preds = []
    val_truth = []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)
            preds = logits.argmax(dim=1).cpu().tolist()

            val_preds.extend(preds)
            val_truth.extend(labels.cpu().tolist())

    val_acc = accuracy_score(val_truth, val_preds)

    if val_acc > best_acc:
        best_acc = val_acc
        best_model_weights = copy.deepcopy(model.state_dict())
        torch.save(best_model_weights, cfg.save_path)

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Best: {best_acc:.4f}"
    )

model.load_state_dict(best_model_weights)

print("Training complete.")
print("Best validation accuracy:", best_acc)

In [ ]:
import torch.nn.functional as F
import cv2
from google.colab import files
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np


def generate_gradcam(model, input_tensor, target_class=None):
    model.eval()

    activations = []
    gradients = []

    target_layer = model.layer4[-1]

    def forward_hook(module, input, output):
        activations.append(output)

    def backward_hook(module, grad_input, grad_output):
        gradients.append(grad_output[0])

    forward_handle = target_layer.register_forward_hook(forward_hook)
    backward_handle = target_layer.register_full_backward_hook(backward_hook)

    input_tensor = input_tensor.clone().detach().requires_grad_(True)

    output = model(input_tensor)

    if target_class is None:
        target_class = output.argmax(dim=1).item()

    model.zero_grad()
    score = output[0, target_class]
    score.backward()

    activation = activations[0].detach()
    gradient = gradients[0].detach()

    weights = gradient.mean(dim=(2, 3), keepdim=True)
    cam = (weights * activation).sum(dim=1, keepdim=True)
    cam = F.relu(cam)

    cam = F.interpolate(
        cam,
        size=(cfg.image_size, cfg.image_size),
        mode="bilinear",
        align_corners=False
    )

    cam = cam.squeeze().cpu().numpy()
    cam = cam - cam.min()
    cam = cam / (cam.max() + 1e-8)

    forward_handle.remove()
    backward_handle.remove()

    return cam, target_class


def predict_with_explanation(image_path):
    test_image = Image.open(image_path).convert("RGB")

    input_tensor = val_transform(test_image).unsqueeze(0).to(device)

    model.eval()

    with torch.no_grad():
        logits = model(input_tensor)
        probs = F.softmax(logits, dim=1)[0].cpu()

    real_prob = float(probs[0])
    ai_prob = float(probs[1])

    predicted_class = 1 if ai_prob > real_prob else 0
    predicted_label = "AI-generated" if predicted_class == 1 else "Real"

    cam, _ = generate_gradcam(model, input_tensor, predicted_class)

    display_image = test_image.resize((cfg.image_size, cfg.image_size))
    display_np = np.array(display_image)

    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)

    overlay = np.uint8(0.55 * display_np + 0.45 * heatmap)

    print("Prediction:", predicted_label)
    print("Real probability:", round(real_prob, 4))
    print("AI-generated probability:", round(ai_prob, 4))

    if max(real_prob, ai_prob) < 0.65:
        print("Confidence level: Low")
    elif max(real_prob, ai_prob) < 0.85:
        print("Confidence level: Medium")
    else:
        print("Confidence level: High")

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 3, 1)
    plt.imshow(display_np)
    plt.title(predicted_label)
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(cam, cmap="jet")
    plt.title("Grad-CAM Heatmap")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(overlay)
    plt.title("Model Focus Area")
    plt.axis("off")

    plt.show()


uploaded = files.upload()
image_path = list(uploaded.keys())[0]

predict_with_explanation(image_path)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

acc = accuracy_score(val_truth, val_preds)
precision = precision_score(val_truth, val_preds)
recall = recall_score(val_truth, val_preds)
f1 = f1_score(val_truth, val_preds)

print("Accuracy:", round(acc * 100, 2), "%")
print("Precision:", round(precision * 100, 2), "%")
print("Recall:", round(recall * 100, 2), "%")
print("F1-score:", round(f1 * 100, 2), "%")